# Project Aura 2.0 — Notebook 07: Deployment Preparation

## Overview

Prepares Project Aura for deployment: configuration, FastAPI backend, Pydantic schemas, health/readiness, WebSocket prep, Docker. Does NOT claim deployment has occurred.

## Sections
1-15. See cells below.

In [ ]:
!pip install --quiet fastapi uvicorn pydantic httpx psutil mediapipe opencv-python-headless numpy pillow matplotlib
print("Dependencies installed for NB07")

## 1. Deployment Overview

## 2. Architecture Inspection

In [ ]:
import os, sys
for p in [os.getcwd(), "/content/project-aura"]:
    if p not in sys.path: sys.path.insert(0, p)
for root, dirs, files in os.walk("src"):
    for f in files:
        if f.endswith(".py"): print(f"  {os.path.join(root, f)}")

## 3. Configuration

In [ ]:
from src.core.config import load_config
from src.api.config import load_api_config
config = load_config(); api_config = load_api_config()
print(f"Model: {config.vision.model_name}, GPU: {config.colab.gpu_enabled}")
print(f"API: {api_config.host}:{api_config.port}, env={api_config.environment}")

## 4. API Schemas

In [ ]:
from src.api.schemas import HealthResponse, ReadinessResponse, ModelInfoResponse, InferenceRequest, InferenceResponse, GestureRequest, GestureResponse, AgentDecideRequest, AgentDecideResponse
for cls in [HealthResponse, ReadinessResponse, ModelInfoResponse, InferenceRequest, InferenceResponse, GestureRequest, GestureResponse, AgentDecideRequest, AgentDecideResponse]:
    print(f"  {cls.__name__}: {len(cls.model_fields)} fields")

## 5. Model Service

In [ ]:
from src.api.services.model_service import ModelService
ms = ModelService(model_name="efficientnet_b0")
print(f"Model: {ms.model_name}, loaded={ms.is_loaded()}")
print(f"Info: {ms.get_model_info()}")

## 6. Inference Route

In [ ]:
from src.api.routes.inference import router as ir
for r in ir.routes: print(f"  {r.methods} {r.path}")

## 7. Gesture Route

In [ ]:
from src.api.routes.gesture import router as gr
for r in gr.routes: print(f"  {r.methods} {r.path}")
from src.api.services.gesture_service import GestureService
print(f"Available: {GestureService().is_available()}")

## 8. Agent Route

In [ ]:
from src.api.routes.agent import router as ar
for r in ar.routes:
    print(f"  {getattr(r,'methods',['WS'])} {r.path}")
from src.api.services.agent_service import AgentService
result = AgentService().decide({"observation_source":"test","observation_type":"gesture","data":{"gesture":"OPEN_PALM"},"confidence":0.9})
print(f"Agent test: {result['selected_action']}")

## 9. Health & Readiness

In [ ]:
from src.api.routes.health import router as hr
for r in hr.routes: print(f"  {r.methods} {r.path}")

## 10. WebSocket Prep

`/ws/gesture` scaffold. Not production WebRTC.

In [ ]:
print("WebSocket: /ws/gesture (scaffold)")
print("Production WebRTC NOT implemented")

## 11. API Testing

In [ ]:
from fastapi.testclient import TestClient
from src.api.main import app
client = TestClient(app)
print(f"GET /health: {client.get('/health').status_code}")
print(f"GET /ready: {client.get('/ready').status_code}")
print(f"GET /model/info: {client.get('/model/info').status_code}")
r = client.post("/agent/decide", json={"observation_source":"test","observation_type":"gesture","data":{"gesture":"OPEN_PALM"},"confidence":0.9})
print(f"POST /agent/decide: {r.status_code} - {r.json()['selected_action']}")
r = client.post("/inference", json={"image_base64":"bad","format":"jpeg"})
print(f"POST /inference (bad): {r.status_code}")
print("All API tests passed")

## 12. Docker Preparation

In [ ]:
dockerfile = "FROM python:3.11-slim\nWORKDIR /app\nRUN apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0 libsm6 libxrender1 libxext6 && rm -rf /var/lib/apt/lists/*\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY src/ ./src/\nEXPOSE 8000\nCMD [\"uvicorn\", \"src.api.main:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8000\"]\n"
reqs = "numpy>=1.24.0\nopencv-python-headless>=4.8.0\npillow>=10.0.0\nfastapi>=0.100.0\nuvicorn>=0.23.0\npydantic>=2.0.0\nmediapipe>=0.10.0\npsutil>=5.9.0\nmatplotlib>=3.7.0\n"
os.makedirs("deploy", exist_ok=True)
open("deploy/Dockerfile","w").write(dockerfile)
open("deploy/requirements.txt","w").write(reqs)
print("Dockerfile + requirements in deploy/ (CPU-compatible)")

## 13. Deployment Instructions

In [ ]:
print("1. pip install -r deploy/requirements.txt")
print("2. cp .env.example .env")
print("3. uvicorn src.api.main:app --host 0.0.0.0 --port 8000")
print("4. pytest tests/test_api.py -v")
print("5. docker build -t project-aura:2.0 -f deploy/Dockerfile .")
print("6. docker run -p 8000:8000 project-aura:2.0")
print("NOTE: NOT deployed to any external service.")

## 14. Validation

In [ ]:
for path, desc in [("deploy/Dockerfile","Dockerfile"),("deploy/requirements.txt","requirements"),("src/api/main.py","FastAPI"),("src/api/config.py","config"),("src/api/schemas/","schemas"),("src/api/routes/","routes"),("src/api/services/","services")]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {desc}")
from src.api.main import app
for r in app.routes:
    if hasattr(r, "methods"): print(f"  {r.methods} {r.path}")

## 15. Final Summary

FastAPI backend, schemas, lazy loading, WebSocket scaffold, Docker prep. NOT deployed.